In [1]:
!pip install pandas numpy scikit-learn nltk flask pyngrok requests

In [2]:
import pandas as pd
import numpy as np
import ast
import pickle
import requests

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
from google.colab import files
uploaded = files.upload()

Saving tmdb_5000_credits.csv to tmdb_5000_credits.csv
Saving tmdb_5000_movies.csv to tmdb_5000_movies.csv


In [4]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

In [5]:
movies.head()
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [6]:
movies = movies.merge(credits, on='title')
movies.shape

(4809, 23)

In [7]:
movies = movies[['movie_id','title','overview',
                 'genres','keywords','cast']]

# Data Cleaning

In [8]:
def convert(text):
    L=[]
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

In [9]:
movies['genres']=movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)

Extraction of top 3 cast

In [10]:
def top_cast(text):
    L=[]
    counter=0

    for i in ast.literal_eval(text):
        if counter != 3:
            L.append(i['name'])
            counter+=1
        else:
            break
    return L

In [11]:
movies['cast']=movies['cast'].apply(top_cast)

In [13]:
movies['overview']=movies['overview'].fillna('').apply(lambda x:x.split())

Removing spaces

In [14]:
for feature in ['genres','keywords','cast']:
    movies[feature]=movies[feature].apply(
        lambda x:[i.replace(" ","") for i in x]
    )

Creating tag columns

In [15]:
movies['tags']=movies['overview'] + \
               movies['genres'] + \
               movies['keywords'] + \
               movies['cast']

In [16]:
new_df = movies[['movie_id','title','tags']]

Convert list to string

In [17]:
new_df['tags']=new_df['tags'].apply(
    lambda x:" ".join(x)
)

/tmp/ipykernel_3493/980557084.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(


In [18]:
new_df['tags']=new_df['tags'].apply(
    lambda x:x.lower()
)

/tmp/ipykernel_3493/366485576.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(


# Vectorisation

In [19]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = tfidf.fit_transform(
    new_df['tags']
).toarray()

Cosine similarity

In [20]:
similarity = cosine_similarity(vectors)
similarity.shape

(4809, 4809)

# Recommendation Function

In [21]:
def recommend(movie):
    movie = movie.lower()

    try:
        idx = new_df[
            new_df['title'].str.lower()==movie
        ].index[0]
    except:
        return "Movie not found"

    distances = similarity[idx]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x:x[1]
    )[1:6]

    result=[]

    for i in movie_list:
        result.append(
            new_df.iloc[i[0]].title
        )

    return result

In [22]:
recommend("Avatar")

['Falcon Rising',
 'Battle: Los Angeles',
 'Apollo 18',
 'Star Trek Into Darkness',
 'Predators']

Movie Poster Fetching

In [23]:
API_KEY = "0aa63bc8602d38e7673c4ab5ef41453c"

In [24]:
def fetch_poster(movie_id):
    url=f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={API_KEY}"
    data=requests.get(url).json()

    return "https://image.tmdb.org/t/p/w500/" + \
           data['poster_path']

In [25]:
def recommend_with_posters(movie):
    idx = new_df[
        new_df['title'].str.lower()==movie.lower()
    ].index[0]

    distances=similarity[idx]

    movie_list=sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x:x[1]
    )[1:6]

    names=[]
    posters=[]

    for i in movie_list:
        movie_id=new_df.iloc[i[0]].movie_id
        names.append(new_df.iloc[i[0]].title)
        posters.append(fetch_poster(movie_id))

    return names, posters

In [26]:
pickle.dump(new_df,
            open('movies.pkl','wb'))

pickle.dump(similarity,
            open('similarity.pkl','wb'))

pickle.dump(tfidf,
            open('vectorizer.pkl','wb'))

In [32]:
!pip install gradio

In [33]:
import gradio as gr

def movie_app(movie):
    return recommend(movie)

demo = gr.Interface(
    fn=movie_app,
    inputs=gr.Textbox(label="Enter movie"),
    outputs=gr.JSON(label="Recommendations"),
    title="Movie Recommendation System"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d46ce6846863c1bbbe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
